# Samaritan — serve a local GGUF from Colab

Puts the model on a rented GPU and leaves your machine to do nothing but
issue HTTP calls. `reason_eval` is ~8 MB resident, so a 4 GB laptop GPU stops
being the bottleneck for an overnight eval.

**Runtime → Change runtime type → L4, then Run all.** The last cell prints the
`SAMARITAN_URL` to paste locally.

**Serves both models at once.** A comparison then runs in a single launch -
no unload, edit, reload between halves - and both models face the same server,
the same batching and the same moment, which is one fewer difference between
them than a sequential pair carries.

It is not less GPU work: the same questions get answered either way. What it
removes is the step in the middle that needs you.

Each model also serves several requests at once (`OLLAMA_NUM_PARALLEL`), so
`ab-eval.ps1 -Shards 3` keeps six streams busy. That is a bigger lever than a
bigger GPU: one stream of a 4B model is bound by memory bandwidth and leaves
most of the card idle, while a batch shares the same weight reads across all of
them.

### Matching the local serving config exactly

The harness pins `temperature`, `max_tokens`, `repeat_penalty` and `seed` in
every request body, so those travel with the client and are already identical.
Two things do **not** travel and must be set on the model itself:

- **`num_ctx 8192`** — Ollama silently *context-shifts* past its window: it
  drops the oldest tokens and returns a normal-looking reply with no error. At
  the 2048 default a 6000-token budget would quietly truncate every long
  answer, and the failures would look like wrong answers rather than clipping.
- **`repeat_penalty 1.0`** — the harness sends this, but `repeat_penalty` is an
  Ollama-native option, not an OpenAI one, so the `/v1` endpoint ignores it and
  would otherwise apply Ollama's own 1.1 default.

Both are set in the Modelfile below. Without them this is not the same
experiment as the local run, however similar the numbers look.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Get the GGUF onto the runtime

Drive is the sane route for 2.4 GB — upload once from your machine, reuse it
every session. `files.upload()` pushes through the browser tab and tends to
die partway at this size.

Put both GGUFs anywhere in Drive. To switch which one is served, change
`MODEL` and re-run this cell and the Modelfile cell - nothing else.

In [ ]:
# Both models, served side by side. Name and exact size travel together so a
# half-finished Drive upload cannot pass as a smaller model - these two files
# differ by 832 bytes out of 2.5 GB, so only an exact count catches a mix-up.
MODELS = {
    'student': ('samaritan-student-v1-185trace-q4_k_m.gguf', 2_497_280_320),
    'base':    ('Qwen3-4B-Thinking-2507-Q4_K_M.gguf',        2_497_281_152),
}

# Drop one to serve only the other - useful when just one has been uploaded.
SERVE = ['student', 'base']
assert all(m in MODELS for m in SERVE), f'SERVE must name entries of {list(MODELS)}'
print('will serve:', ', '.join(SERVE))

import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')

LOCAL = {}
for model in SERVE:
    name, expected = MODELS[model]
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, (f'{name} not found in Drive. Upload it to MyDrive (any '
                  f'folder), or drop {model!r} from SERVE, then re-run.')
    src = hits[0]
    got = os.path.getsize(src)

    # An incomplete upload is what this check exists for: Drive lists a
    # partial file at its partial size with no error, and a truncated GGUF is
    # not a smaller model, it is a corrupt one. Exact byte count, because 99%
    # uploaded is just as broken as 2% and looks far more plausible.
    if got != expected:
        raise AssertionError(
            f'{model}: upload incomplete - {got:,} of {expected:,} bytes '
            f'({got/expected:.1%}). Wait for Drive to finish; it should read '
            f'{expected/1e9:.2f} GB. Then re-run this cell.')
    with open(src, 'rb') as f:
        assert f.read(4) == b'GGUF', f'{model}: not a GGUF file - wrong file?'

    # Copy to local disk: serving off the Drive FUSE mount is slow and drops
    # out on long runs. Named per model, so a same-size successor cannot be
    # silently inherited from a previous session.
    dst = f'/content/{model}.gguf'
    if not os.path.exists(dst) or os.path.getsize(dst) != got:
        shutil.copy(src, dst)
    assert os.path.getsize(dst) == expected, f'{model}: local copy truncated'
    LOCAL[model] = dst
    print(f'{model:<8} {got/1e9:.2f} GB  verified  ->  {dst}')

## 2. Ollama

In [ ]:
import os, subprocess, time, urllib.request, json, shutil, glob

# zstd first: Ollama's installer unpacks with it and Colab does not ship it.
# Without this the install half-succeeds - the binary lands but the runner
# does not - and the failure surfaces later as a model that will not load,
# which reads like a bad GGUF rather than a missing package.
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
!curl -fsSL https://ollama.com/install.sh | sh

# Check the RUNNER, not just the launcher. `ollama` on PATH proves the client
# installed; the thing that actually serves the model is llama-server, and it
# is what goes missing when the unpack fails.
assert shutil.which('ollama'), 'ollama install failed (see output above)'
runner = glob.glob('/usr/local/lib/ollama/**/llama-server', recursive=True)
assert runner, ('ollama installed but its runner did not unpack - almost always '
                'a missing zstd. Re-run this cell.')
print('ollama + runner OK:', runner[0])

subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(2)
# NUM_PARALLEL is what makes sharded evaluation worth anything: without it
# Ollama answers one request at a time and the extra workers just queue.
# MAX_LOADED_MODELS keeps both models resident instead of swapping one out
# for the other on every alternating request, which would be far slower than
# running them sequentially.
#
# KV cache is num_ctx * NUM_PARALLEL * models. At 20480 x 3 x 2 that is ~8.4 GB
# on top of ~5 GB of weights - comfortable on a 24 GB L4. Raising NUM_PARALLEL
# to 4 costs ~2.8 GB more and still fits; the cell below prints the figure.
NUM_PARALLEL = 3
env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '-1',
       'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0',
       'OLLAMA_NUM_PARALLEL': str(NUM_PARALLEL),
       'OLLAMA_MAX_LOADED_MODELS': str(len(SERVE))}
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'),
                       stderr=subprocess.STDOUT, env=env)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=5)
        print('daemon up'); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit('daemon did not start:\n' + open('ollama.log').read()[-2000:])

## 3. Register the model under the alias the harness expects

Each model gets its own alias — `samaritan-student`, `samaritan-base` — so
both can be served at once and a local run says which it wants. `ab-eval.ps1`
passes these through `-RemoteAliases`, matched position by position to
`-Models`.

In [ ]:
# Must match ab-eval.ps1's -Ctx, and exceed MaxTokens + prompt. 8192 was
# too small once the eval budget moved to 16000: the server would have
# context-shifted mid-answer with no error, which scores exactly like a
# wrong answer. See docs - the cap sitting near median demand is what made
# two runs of the same weights disagree on 18 of 40 items.
CTX = 20480
REPEAT_PENALTY = 1.0  # harness default; /v1 ignores the request field

ALIASES = {}
for model in SERVE:
    alias = f'samaritan-{model}'
    with open('Modelfile', 'w') as f:
        f.write(f'FROM {LOCAL[model]}\n'
                f'PARAMETER num_ctx {CTX}\n'
                f'PARAMETER repeat_penalty {REPEAT_PENALTY}\n')
    subprocess.run(['ollama', 'create', alias, '-f', 'Modelfile'], check=True)
    ALIASES[model] = alias
    print(f'  {alias} <- {LOCAL[model]}')

# Print the memory picture rather than discovering the ceiling as an OOM
# halfway through a paid run.
gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total',
                      '--format=csv,noheader'], capture_output=True,
                     text=True).stdout.strip()
print(f'\ncontext budget: {CTX} x {NUM_PARALLEL} parallel x {len(SERVE)} model(s) '
      f'= {CTX * NUM_PARALLEL * len(SERVE):,} tokens of KV')
print(f'gpu: {gpu}')

lst = subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout
for alias in ALIASES.values():
    assert alias in lst, f'{alias} not created:\n' + lst

# Warm every alias. The first request loads the weights, so doing it here keeps
# that 30-60s out of the first evaluated item - and proves each one generates
# before any of them is measured.
for model, alias in ALIASES.items():
    body = json.dumps({'model': alias,
        'messages': [{'role': 'user', 'content': 'What is 2+2? Reply with just the number.'}],
        'stream': False, 'options': {'num_predict': 64}}).encode()
    req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=body,
        headers={'Content-Type': 'application/json'})
    r = json.loads(urllib.request.urlopen(req, timeout=600).read())
    print(f'[OK] {alias} generates:', r['message']['content'][:60].replace(chr(10), ' '))

### Confirm the context actually took

Worth thirty seconds: a `num_ctx` that silently stayed at 2048 produces a run
that looks fine and measures nothing, which is the expensive kind of wrong.

In [ ]:
log = open('ollama.log').read()
import re
hits = re.findall(r'n_ctx\s*=\s*(\d+)', log)
print('n_ctx seen in the server log:', hits[-5:] if hits else '(none logged yet)')
if hits:
    got = max(int(h) for h in hits)
    print(f'{"OK" if got >= CTX else "MISMATCH"}: server built a {got}-token context '
          f'(asked for {CTX})')
    assert got >= CTX, 'context smaller than requested - fix before evaluating'

## 4. Public tunnel

In [ ]:
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
try:
    cf.terminate()
except Exception:
    pass
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434',
                       '--http-host-header', 'localhost:11434'],
                      stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
public = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m:
        public = m.group(0); break
assert public, 'no tunnel URL - cf.log:\n' + open('cf.log').read()[-1000:]

def _get(url, timeout=60):
    req = urllib.request.Request(url, headers={'Authorization': 'Bearer ollama'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read().decode()

ok = False
for _ in range(8):
    try:
        s, b = _get(public + '/v1/models')
        if s == 200 and all(a in b for a in ALIASES.values()):
            ok = True; break
    except Exception:
        pass
    time.sleep(5)
assert ok, (f'tunnel up at {public} but /v1/models is missing one of '
            f'{list(ALIASES.values())} - re-run the alias cell')

models = {'student': 'student-v1-q4km', 'base': 'base-q4km'}
keys = ','.join(models[m] for m in SERVE)
aliases = ','.join(ALIASES[m] for m in SERVE)
print('\n' + '=' * 78)
print('Run this locally, from the repo root (PowerShell). One launch, both models:')
print()
# One line, no continuation character: PowerShell continues with a backtick
# rather than a backslash, and a trailing space after either one silently
# breaks the command. A single line cannot be got wrong by pasting.
cmd = ('powershell -ExecutionPolicy Bypass -File scripts/ab-eval.ps1'
       f' -RemoteUrl "{public}/v1"'
       f' -Models {keys}'
       f' -RemoteAliases {aliases}'
       f' -Dataset "$env:USERPROFILE/models/reasoning/generated-d3-s883.jsonl"'
       f' -Limit 40 -Tag "-d3big" -Shards {NUM_PARALLEL} -NoStream')
print(cmd)
print()
print('MaxTokens/Ctx/Seed are correct by default now - leave them off.')
print('=' * 78)

## 5. Keep-alive

Leave this running while you evaluate. Stop it when done — the runtime bills
for as long as it is allocated, whether or not it is busy.

In [ ]:
from datetime import datetime
UNITS_PER_HOUR = 5.3
MAX_HOURS = 8          # 0 = no cap

start = time.time()
beat = 0
while True:
    beat += 1
    hrs = (time.time() - start) / 3600
    if MAX_HOURS and hrs > MAX_HOURS:
        print(f'\nMAX_HOURS ({MAX_HOURS}) reached - stopping keep-alive.')
        break
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,utilization.gpu',
                          '--format=csv,noheader'], capture_output=True,
                         text=True).stdout.strip()
    alive = 'up' if cf.poll() is None else 'DOWN'
    print(f'[{datetime.now():%H:%M:%S}] beat {beat:>4} | gpu {gpu} | tunnel {alive} '
          f'| {hrs:.2f} h | ~{hrs * UNITS_PER_HOUR:.1f} units', flush=True)
    time.sleep(60)